In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from newspaper import Article
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, OrdinalEncoder, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
data = pd.read_csv('data_visualization.csv')

The goal for this file is to repeat the creation of my dashboard while taking into account all the changes we discussed listed in ProjectIteration_Diary.txt. Note I relied on LLMS to make my plots interactable and styled.

In [44]:
data = data.dropna(subset=["ticker", "publisher"])
data["publisher"] = data["publisher"].astype(str).str.strip()
data["ticker"] = data["ticker"].astype(str).str.strip()

numeric_cols = [
    "sentiment_encoded",
    "return_30d",
    "return_6m",
    "long_term",
    "accuracy"
]

for col in numeric_cols:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors="coerce")

data["accuracy"] = data["accuracy"].fillna(0)
data["sentiment_encoded"] = data["sentiment_encoded"].fillna(0)
data["long_term"] = data["long_term"].fillna(0)

In [45]:
display(HTML("""
<style>
    .dashboard-wrap {
        background: #0b0f14;
        color: #f4f7fb;
        padding: 22px;
        border-radius: 16px;
        border: 1px solid #1f2937;
        font-family: Inter, Arial, sans-serif;
    }

    .dashboard-title {
        font-size: 30px;
        font-weight: 700;
        letter-spacing: 0.4px;
        margin-bottom: 4px;
        color: #f9fafb;
    }

    .dashboard-subtitle {
        font-size: 14px;
        color: #9ca3af;
        margin-bottom: 22px;
    }

    .section-title {
        font-size: 18px;
        font-weight: 700;
        color: #f9fafb;
        margin-top: 16px;
        margin-bottom: 12px;
        letter-spacing: 0.2px;
    }

    .metric-grid {
        display: grid;
        grid-template-columns: repeat(4, minmax(180px, 1fr));
        gap: 12px;
        margin-bottom: 16px;
    }

    .metric-card {
        background: linear-gradient(180deg, #111827 0%, #0f172a 100%);
        border: 1px solid #1f2937;
        border-radius: 14px;
        padding: 14px 16px;
        box-shadow: 0 6px 18px rgba(0,0,0,0.28);
    }

    .metric-label {
        font-size: 12px;
        color: #9ca3af;
        margin-bottom: 8px;
        text-transform: uppercase;
        letter-spacing: 0.5px;
    }

    .metric-value {
        font-size: 22px;
        font-weight: 700;
        color: #f9fafb;
    }

    .metric-note {
        font-size: 12px;
        color: #34d399;
        margin-top: 6px;
    }

    .summary-box {
        background: linear-gradient(180deg, #111827 0%, #0f172a 100%);
        border: 1px solid #1f2937;
        border-radius: 14px;
        padding: 16px;
        margin-bottom: 16px;
    }

    .summary-title {
        font-size: 16px;
        font-weight: 700;
        color: #f9fafb;
        margin-bottom: 8px;
    }

    .summary-text {
        font-size: 14px;
        color: #d1d5db;
        line-height: 1.6;
    }

    .controls-row {
        margin-bottom: 18px;
    }

    table.dark-sortable {
        width: 100%;
        border-collapse: collapse;
        font-size: 13px;
        margin-top: 8px;
        background: #0f172a;
        color: #f3f4f6;
        border-radius: 12px;
        overflow: hidden;
    }

    table.dark-sortable th,
    table.dark-sortable td {
        border-bottom: 1px solid #1f2937;
        padding: 10px 12px;
        text-align: left;
    }

    table.dark-sortable th {
        background: #111827;
        color: #e5e7eb;
        cursor: pointer;
        position: sticky;
        top: 0;
        z-index: 1;
        user-select: none;
    }

    table.dark-sortable tr:hover td {
        background: #111827;
    }

    .table-wrap {
        max-height: 420px;
        overflow-y: auto;
        border: 1px solid #1f2937;
        border-radius: 12px;
        margin-bottom: 16px;
    }

    .small-note {
        font-size: 12px;
        color: #9ca3af;
        margin-top: 6px;
        margin-bottom: 12px;
    }

    .widget-label {
        color: #f3f4f6 !important;
        font-weight: 600;
    }
</style>
"""))


In [46]:
ticker_features = data.groupby("ticker").agg(
    avg_sentiment=("sentiment_encoded", "mean"),
    sentiment_std=("sentiment_encoded", "std"),
    bullish_rate=("sentiment_encoded", lambda x: (x == 1).mean()),
    bearish_rate=("sentiment_encoded", lambda x: (x == -1).mean()),
    neutral_rate=("sentiment_encoded", lambda x: (x == 0).mean()),
    avg_return_30d=("return_30d", "mean"),
    avg_return_6m=("return_6m", "mean"),
    vol_30d=("return_30d", "std"),
    article_count=("headline", "count"),
    long_term_ratio=("long_term", "mean")
).reset_index()

ticker_features["sentiment_std"] = ticker_features["sentiment_std"].fillna(0)
ticker_features["vol_30d"] = ticker_features["vol_30d"].fillna(0)
ticker_features["log_article_count"] = np.log1p(ticker_features["article_count"])

In [47]:
publisher_overall = data.groupby("publisher").agg(
    overall_articles=("headline", "count"),
    overall_accuracy=("accuracy", "mean"),
    overall_avg_sentiment=("sentiment_encoded", "mean"),
    overall_long_term_ratio=("long_term", "mean"),
    unique_tickers=("ticker", "nunique")
).reset_index()

publisher_overall["log_overall_articles"] = np.log1p(publisher_overall["overall_articles"])

publisher_by_ticker = data.groupby(["ticker", "publisher"]).agg(
    ticker_articles=("headline", "count"),
    ticker_accuracy=("accuracy", "mean"),
    ticker_avg_sentiment=("sentiment_encoded", "mean"),
    ticker_long_term_ratio=("long_term", "mean")
).reset_index()

publisher_lookup = publisher_by_ticker.merge(
    publisher_overall,
    on="publisher",
    how="left"
)


In [48]:
publisher_cluster_cols = [
    "overall_accuracy",
    "overall_avg_sentiment",
    "log_overall_articles",
    "unique_tickers",
    "overall_long_term_ratio"
]

publisher_X = publisher_overall[publisher_cluster_cols].fillna(0)

publisher_scaler = StandardScaler()
publisher_X_scaled = publisher_scaler.fit_transform(publisher_X)

publisher_kmeans = KMeans(n_clusters=4, random_state=42, n_init=20)
publisher_overall["cluster"] = publisher_kmeans.fit_predict(publisher_X_scaled)

publisher_silhouette = silhouette_score(publisher_X_scaled, publisher_overall["cluster"])

publisher_pca = PCA(n_components=2, random_state=42)
publisher_pca_vals = publisher_pca.fit_transform(publisher_X_scaled)

publisher_overall["pca1"] = publisher_pca_vals[:, 0]
publisher_overall["pca2"] = publisher_pca_vals[:, 1]

publisher_lookup = publisher_lookup.merge(
    publisher_overall[["publisher", "cluster", "pca1", "pca2"]],
    on="publisher",
    how="left"
)

cluster_summary = publisher_overall.groupby("cluster").agg(
    overall_accuracy=("overall_accuracy", "mean"),
    overall_articles=("overall_articles", "mean"),
    overall_avg_sentiment=("overall_avg_sentiment", "mean"),
    unique_tickers=("unique_tickers", "mean")
).reset_index()

accuracy_median = publisher_overall["overall_accuracy"].median()
articles_median = publisher_overall["overall_articles"].median()
sentiment_median = publisher_overall["overall_avg_sentiment"].median()
coverage_median = publisher_overall["unique_tickers"].median()

cluster_descriptions = {}

for _, row in cluster_summary.iterrows():
    cluster_id = int(row["cluster"])

    parts = []

    if row["overall_accuracy"] >= accuracy_median:
        parts.append("higher accuracy")
    else:
        parts.append("lower accuracy")

    if row["overall_articles"] >= articles_median:
        parts.append("high volume")
    else:
        parts.append("lower volume")

    if row["unique_tickers"] >= coverage_median:
        parts.append("broad coverage")
    else:
        parts.append("more specialized coverage")

    if row["overall_avg_sentiment"] >= sentiment_median:
        parts.append("more positive sentiment")
    else:
        parts.append("more neutral/negative sentiment")

    cluster_descriptions[cluster_id] = ", ".join(parts).capitalize() + "."

